# Initializa Notebook

In [ ]:
%env WORKDIR = '/tmp/vault'
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = next((directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()), None)
if ENV_FILE is None:
    raise FileNotFoundError("Could not find the persistent .env file")
load_dotenv(ENV_FILE)

VAULT_TOKEN = os.getenv('VAULT_TOKEN')
VAULT_ADDR = os.getenv('VAULT_ADDR')
VAULT_CACERT = os.getenv('VAULT_CACERT')

In [ ]:
!doormat login -f

import os
import subprocess

# Import the credentials produced by doormat into the notebook kernel.
result = subprocess.run(
    ["bash", "-lc", 'eval "$(doormat aws -a aws_jose.merchan_test export)" && env -0'],
    check=True,
    capture_output=True,
)
for entry in result.stdout.split(b"\0"):
    if entry.startswith(b"AWS_") and b"=" in entry:
        key, value = entry.split(b"=", 1)
        os.environ[key.decode()] = value.decode()

os.environ["AWS_REGION"] = "eu-central-1"

# Create Oracle Database in K8S

In [ ]:
%%bash
set -euo pipefail

export ORACLE_REG_EMAIL=jose.merchan@hashicorp.com
export ORACLE_REG_PASSWORD="codwir-zocDe1-vammyv"

ORACLE_NAMESPACE=oracle
ORACLE_DB_SECRET=oracle-db-credentials

kubectl create namespace "$ORACLE_NAMESPACE" --dry-run=client -o yaml | kubectl apply -f -

# No se reemplazan contraseñas si el PVC ya contiene una base inicializada.
if ! kubectl get secret "$ORACLE_DB_SECRET" -n "$ORACLE_NAMESPACE" >/dev/null 2>&1; then
  SYS_PASSWORD="OraSys1$(openssl rand -hex 12)"
  VAULT_DB_PASSWORD="OraVault1$(openssl rand -hex 12)"
  kubectl create secret generic "$ORACLE_DB_SECRET" \
    --namespace="$ORACLE_NAMESPACE" \
    --from-literal=sys-password="$SYS_PASSWORD" \
    --from-literal=vault-password="$VAULT_DB_PASSWORD"
fi

echo "Namespace y credenciales de la base de datos preparados."

## Oracle Database Free

Se usa la imagen pública ligera `database/free:latest-lite` sobre `linux/amd64`, sin credenciales de registro. La imagen personalizada de Vault contiene Instant Client 19.26 y 23.26.3; el plugin oficial carga IC23, compatible tanto con Oracle Database Free como con Oracle Database 19c. Vault Enterprise exige conservar el nombre firmado `vault-plugin-database-oracle`, por lo que no se registran alias de plugin sin firma. La conexión se realiza contra la PDB `FREEPDB1` mediante el servicio interno `oracle-db.oracle.svc.cluster.local:1521`.

In [ ]:
%%bash
set -euo pipefail

if ! kubectl get nodes -o json | jq -e 'any(.items[]; .status.nodeInfo.architecture == "amd64")' >/dev/null; then
  echo "ERROR: se necesita al menos un nodo linux/amd64 para esta imagen de Oracle Database." >&2
  exit 1
fi

kubectl apply -f - <<'YAML'
apiVersion: v1
kind: Service
metadata:
  name: oracle-db
  namespace: oracle
  labels:
    app.kubernetes.io/name: oracle-db
spec:
  type: ClusterIP
  selector:
    app.kubernetes.io/name: oracle-db
  ports:
    - name: oracle-listener
      port: 1521
      targetPort: 1521
---
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: oracle-db
  namespace: oracle
spec:
  serviceName: oracle-db
  replicas: 1
  selector:
    matchLabels:
      app.kubernetes.io/name: oracle-db
  template:
    metadata:
      labels:
        app.kubernetes.io/name: oracle-db
    spec:
      nodeSelector:
        kubernetes.io/os: linux
        kubernetes.io/arch: amd64
      securityContext:
        fsGroup: 54321
        fsGroupChangePolicy: OnRootMismatch
      terminationGracePeriodSeconds: 120
      containers:
        - name: oracle-db
          image: container-registry.oracle.com/database/free:latest-lite
          imagePullPolicy: IfNotPresent
          securityContext:
            runAsNonRoot: true
            runAsUser: 54321
            runAsGroup: 54321
            allowPrivilegeEscalation: false
          env:
            - name: ORACLE_SID
              value: FREE
            - name: ORACLE_PDB
              value: FREEPDB1
            - name: ORACLE_PWD
              valueFrom:
                secretKeyRef:
                  name: oracle-db-credentials
                  key: sys-password
            - name: ORACLE_CHARACTERSET
              value: AL32UTF8
            - name: INIT_SGA_SIZE
              value: "1024"
            - name: INIT_PGA_SIZE
              value: "256"
          ports:
            - name: listener
              containerPort: 1521
          resources:
            requests:
              cpu: "1"
              memory: 2Gi
            limits:
              cpu: "2"
              memory: 3Gi
          startupProbe:
            exec:
              command: ["/bin/bash", "-c", "/opt/oracle/checkDBStatus.sh"]
            periodSeconds: 15
            timeoutSeconds: 10
            failureThreshold: 120
          readinessProbe:
            exec:
              command: ["/bin/bash", "-c", "/opt/oracle/checkDBStatus.sh"]
            periodSeconds: 15
            timeoutSeconds: 10
            failureThreshold: 4
          livenessProbe:
            exec:
              command: ["/bin/bash", "-c", "/opt/oracle/checkDBStatus.sh"]
            periodSeconds: 30
            timeoutSeconds: 10
            failureThreshold: 6
          volumeMounts:
            - name: oracle-data
              mountPath: /opt/oracle/oradata
            - name: dshm
              mountPath: /dev/shm
      volumes:
        - name: dshm
          emptyDir:
            medium: Memory
            sizeLimit: 1Gi
  volumeClaimTemplates:
    - metadata:
        name: oracle-data
      spec:
        accessModes: ["ReadWriteOnce"]
        resources:
          requests:
            storage: 50Gi
YAML

kubectl rollout status statefulset/oracle-db -n oracle --timeout=30m

## Usuario técnico para Vault

El siguiente bloque crea un usuario dedicado dentro de `FREEPDB1`. La contraseña permanece en el Secret de Kubernetes y no se muestra en la salida.

In [ ]:
%%bash
set -euo pipefail

VAULT_DB_PASSWORD=$(kubectl get secret oracle-db-credentials -n oracle -o jsonpath='{.data.vault-password}' | base64 --decode)

kubectl exec -i -n oracle oracle-db-0 -- /bin/bash -s -- "$VAULT_DB_PASSWORD" <<'CONTAINER_SCRIPT'
set -euo pipefail
VAULT_DB_PASSWORD=$1
sqlplus -s / as sysdba <<SQL
WHENEVER SQLERROR EXIT SQL.SQLCODE
ALTER SESSION SET CONTAINER=FREEPDB1;
DECLARE
  user_count NUMBER;
BEGIN
  SELECT COUNT(*) INTO user_count FROM dba_users WHERE username = 'VAULT';
  IF user_count = 0 THEN
    EXECUTE IMMEDIATE 'CREATE USER VAULT IDENTIFIED BY "${VAULT_DB_PASSWORD}"';
  ELSE
    EXECUTE IMMEDIATE 'ALTER USER VAULT IDENTIFIED BY "${VAULT_DB_PASSWORD}" ACCOUNT UNLOCK';
  END IF;
END;
/
GRANT CREATE USER TO VAULT WITH ADMIN OPTION;
GRANT ALTER USER TO VAULT WITH ADMIN OPTION;
GRANT DROP USER TO VAULT WITH ADMIN OPTION;
GRANT CONNECT TO VAULT WITH ADMIN OPTION;
GRANT CREATE SESSION TO VAULT WITH ADMIN OPTION;
GRANT SELECT ON SYS.GV_\$SESSION TO VAULT;
GRANT SELECT ON SYS.V_\$SQL TO VAULT;
GRANT ALTER SYSTEM TO VAULT WITH ADMIN OPTION;
EXIT
SQL

# Verifica las mismas credenciales mediante Oracle Net antes de configurar Vault.
sqlplus -L -s /nolog <<SQL
WHENEVER SQLERROR EXIT SQL.SQLCODE
CONNECT VAULT/"${VAULT_DB_PASSWORD}"@//127.0.0.1:1521/FREEPDB1
SELECT 'VAULT_LOGIN_OK' FROM dual;
EXIT
SQL
CONTAINER_SCRIPT

echo "Usuario técnico VAULT creado y autenticación Oracle Net validada en FREEPDB1."

## Conectar Vault y validar credenciales dinámicas

In [ ]:
%%bash
set -euo pipefail

VAULT_DB_PASSWORD=$(kubectl get secret oracle-db-credentials -n oracle -o jsonpath='{.data.vault-password}' | base64 --decode)
if [[ -z "${VAULT_TOKEN:-}" && -f "${WORKDIR}/cluster-keys.json" ]]; then
  VAULT_TOKEN=$(jq -r '.root_token' "${WORKDIR}/cluster-keys.json")
fi
: "${VAULT_ADDR:?No se encontró VAULT_ADDR en el entorno}"
: "${VAULT_TOKEN:?No se encontró VAULT_TOKEN ni ${WORKDIR}/cluster-keys.json}"
export VAULT_ADDR VAULT_TOKEN
if [[ -n "${VAULT_CACERT:-}" ]]; then
  export VAULT_CACERT
fi

vault status >/dev/null
if ! vault secrets list -format=json | jq -e 'has("database/")' >/dev/null; then
  vault secrets enable database
fi

vault write database/config/oracle \
  plugin_name=vault-plugin-database-oracle \
  allowed_roles=oracle-dynamic,oracle-static \
  connection_url='{{username}}/{{password}}@//oracle-db.oracle.svc.cluster.local:1521/FREEPDB1' \
  username=VAULT \
  password="$VAULT_DB_PASSWORD" \
  verify_connection=true



In [ ]:
%%bash
vault write database/roles/oracle-dynamic \
  db_name=oracle \
  creation_statements='CREATE USER {{username}} IDENTIFIED BY "{{password}}"; GRANT CONNECT TO {{username}}; GRANT CREATE SESSION TO {{username}};' \
  default_ttl=1h \
  max_ttl=24h


In [ ]:
! vault read -format=json database/creds/oracle-dynamic


## Validación

In [ ]:
%%bash
set -euo pipefail

STATIC_JSON=$(vault read -format=json database/creds/oracle-dynamic)
STATIC_USER=$(jq -r '.data.username' <<<"$STATIC_JSON")
STATIC_PASSWORD=$(jq -r '.data.password' <<<"$STATIC_JSON")

LOGIN_RESULT=$(kubectl exec -i -n oracle oracle-db-0 -- /bin/bash -s -- "$STATIC_USER" "$STATIC_PASSWORD" <<'CONTAINER_SCRIPT'
set -euo pipefail
sqlplus -L -s /nolog <<SQL
WHENEVER SQLERROR EXIT SQL.SQLCODE
CONNECT $1/"$2"@//127.0.0.1:1521/FREEPDB1
SET HEADING OFF FEEDBACK OFF PAGES 0
SELECT 'STATIC_LOGIN_OK' FROM dual;
EXIT
SQL
CONTAINER_SCRIPT
)

jq '{username:.data.username, password:.data.password, lease_duration:.lease_duration, renewable:.renewable}' <<<"$STATIC_JSON"
echo "Login del dynamic role: $(tr -d '[:space:]' <<<"$LOGIN_RESULT")"

# Static Role

In [ ]:
%%bash
set -euo pipefail

# No se restablece fuera de Vault una cuenta que ya está siendo gestionada.
if ! vault read database/static-roles/oracle-static >/dev/null 2>&1; then
  INITIAL_STATIC_PASSWORD="OraStatic1$(openssl rand -hex 12)"
  kubectl exec -i -n oracle oracle-db-0 -- /bin/bash -s -- "$INITIAL_STATIC_PASSWORD" <<'CONTAINER_SCRIPT'
set -euo pipefail
INITIAL_STATIC_PASSWORD=$1
sqlplus -s / as sysdba <<SQL
WHENEVER SQLERROR EXIT SQL.SQLCODE
ALTER SESSION SET CONTAINER=FREEPDB1;
DECLARE
  user_count NUMBER;
BEGIN
  SELECT COUNT(*) INTO user_count FROM dba_users WHERE username='VAULT_STATIC';
  IF user_count=0 THEN
    EXECUTE IMMEDIATE 'CREATE USER VAULT_STATIC IDENTIFIED BY "${INITIAL_STATIC_PASSWORD}"';
  ELSE
    EXECUTE IMMEDIATE 'ALTER USER VAULT_STATIC IDENTIFIED BY "${INITIAL_STATIC_PASSWORD}" ACCOUNT UNLOCK';
  END IF;
END;
/
GRANT CREATE SESSION TO VAULT_STATIC;
EXIT
SQL
CONTAINER_SCRIPT

  vault write database/static-roles/oracle-static \
    db_name=oracle \
    username=VAULT_STATIC \
    rotation_period=24h >/dev/null
fi

vault read -format=json database/static-roles/oracle-static | \
  jq '{db_name:.data.db_name, username:.data.username, rotation_period:.data.rotation_period}'

## Validación

In [ ]:
%%bash
set -euo pipefail

STATIC_JSON=$(vault read -format=json database/static-creds/oracle-static)
STATIC_USER=$(jq -r '.data.username' <<<"$STATIC_JSON")
STATIC_PASSWORD=$(jq -r '.data.password' <<<"$STATIC_JSON")

LOGIN_RESULT=$(kubectl exec -i -n oracle oracle-db-0 -- /bin/bash -s -- "$STATIC_USER" "$STATIC_PASSWORD" <<'CONTAINER_SCRIPT'
set -euo pipefail
sqlplus -L -s /nolog <<SQL
WHENEVER SQLERROR EXIT SQL.SQLCODE
CONNECT $1/"$2"@//127.0.0.1:1521/FREEPDB1
SET HEADING OFF FEEDBACK OFF PAGES 0
SELECT 'STATIC_LOGIN_OK' FROM dual;
EXIT
SQL
CONTAINER_SCRIPT
)

jq '{username:.data.username, ttl:.data.ttl, rotation_period:.data.rotation_period}' <<<"$STATIC_JSON"
echo "Login del static role: $(tr -d '[:space:]' <<<"$LOGIN_RESULT")"

## Clean UP

In [ ]:
%%bash
set -euo pipefail

: "${VAULT_ADDR:?No se encontró VAULT_ADDR en el entorno}"
: "${VAULT_TOKEN:?No se encontró VAULT_TOKEN en el entorno}"

# Revocar primero las credenciales dinámicas y cualquier lease restante del mount.
vault lease revoke -prefix database/creds/oracle-dynamic/ 2>/dev/null || true
vault lease revoke -prefix database/ 2>/dev/null || true

# Eliminar roles y conexión antes de desmontar el secrets engine.
vault delete database/roles/oracle-dynamic 2>/dev/null || true
vault delete database/static-roles/oracle-static 2>/dev/null || true
vault delete database/config/oracle 2>/dev/null || true
if vault secrets list -format=json | jq -e 'has("database/")' >/dev/null; then
  vault secrets disable database/
fi

# Los usuarios estáticos son externos a Vault y se eliminan explícitamente.
if kubectl get pod oracle-db-0 -n oracle >/dev/null 2>&1; then
  kubectl exec -i -n oracle oracle-db-0 -- /bin/bash -s <<'CONTAINER_SCRIPT'
set -euo pipefail
sqlplus -s / as sysdba <<'SQL'
WHENEVER SQLERROR EXIT SQL.SQLCODE
ALTER SESSION SET CONTAINER=FREEPDB1;
BEGIN
  FOR managed_user IN (
    SELECT username FROM dba_users
    WHERE username IN ('VAULT_STATIC', 'VAULT') OR username LIKE 'V_ROOT_ORACLE_D_%'
  ) LOOP
    EXECUTE IMMEDIATE 'DROP USER "' || managed_user.username || '" CASCADE';
  END LOOP;
END;
/
EXIT
SQL
CONTAINER_SCRIPT
fi

# Oracle y el PVC se conservan salvo que se solicite explícitamente su eliminación.
DELETE_ORACLE_K8S=${DELETE_ORACLE_K8S:-false}
DELETE_ORACLE_DATA=${DELETE_ORACLE_DATA:-false}
if [[ "$DELETE_ORACLE_K8S" == "true" ]]; then
  kubectl delete statefulset/oracle-db service/oracle-db secret/oracle-db-credentials \
    -n oracle --ignore-not-found
fi
if [[ "$DELETE_ORACLE_DATA" == "true" ]]; then
  kubectl delete pvc/oracle-data-oracle-db-0 -n oracle --ignore-not-found
fi

echo "Cleanup de Vault completado. Oracle/PVC conservados salvo petición explícita."